# Music Annotation Project V7.9: Coarse-to-Fine Observed-Ink Preservation Challenger

**Notebook 22. Runtime:** CPU. Select **Runtime > Run all**.

This notebook develops a V7.9 preservation challenger without changing Notebook 09 and without reading frozen V7.8 independent outcomes.

## Search architecture

1. Reproduce frozen V7.8 exactly on all controlled development pages.
2. Select a deterministic development discovery subset using stable SHA-256 page hashes.
3. Run a cheap **route-first coarse search** on that subset.
4. Build a local **fine search** around the best coarse parameter regions.
5. Confirm shortlisted routes across **all controlled development pages**.
6. Perform full reconstruction and diagnostic crops for **only the final three candidates**, plus frozen V7.8.

Clean truth is used only after route generation for development scoring. Pixels outside the removal mask remain immutable. V7.7F is not applied because V7.9 changes preserve routing semantics.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy matplotlib psutil


## Configuration and data firewall


In [ ]:
import cv2, json, hashlib, gc, os, psutil, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime, timezone

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
V75_ROOT=ROOT/'reconstruction_geometry_benchmark_v7_5'
OUTPUT_ROOT=ROOT/'observed_ink_preservation_challenger_v7_9'
V75_RUN_OVERRIDE=None
PAGE_ALLOWLIST=None
MAX_PAGES=None

MASK_DILATION=2
MIN_COMPONENT_AREA=20
GEOMETRY_THRESHOLD=80
BAND_WIDTH=4
DISCOVERY_PAGE_COUNT=16
COARSE_KEEP=12
FINE_KEEP=12
FINALIST_COUNT=3
MAX_DIAGNOSTIC_COMPONENTS=30

# Coarse discovery grid: 3 x 2 x 2 x 2 x 4 = 96 configurations.
COARSE_DARK=[100,140,180]
COARSE_STROKE=[2.5,4.0]
COARSE_MIN_AREA=[3,12]
COARSE_MAX_AREA=[650,1800]
MODES=['connected_dark','edge_stroke','consensus','union']

SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
PROHIBITED_PATH_TOKENS=[
 'independent_structural_validation_v7_8_builder',
 'independent_structural_validation_v7_8_rebuilt',
 'independent_policy_validation_v7_8',
]

def assert_development_path(path):
    text=str(Path(path)).lower()
    if any(token in text for token in PROHIBITED_PATH_TOKENS):
        raise RuntimeError(f'PROHIBITED V7.8 independent path accessed: {path}')

def latest_complete(root,required):
    assert_development_path(root);hits=[]
    if root.exists():
        for p in root.rglob('*'):
            if p.is_dir() and all((p/r).exists() for r in required):hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

def rss_gb():return psutil.Process(os.getpid()).memory_info().rss/(1024**3)
def stable_page_key(page):return hashlib.sha256(('v79-discovery:'+str(page)).encode()).hexdigest()
def candidate_name(mode,p):return f"v79_{mode}_d{p['dark_threshold']}_w{p['stroke_width_limit']}_a{p['min_area']}_{p['max_area']}"
def parse_candidate(name):
    m=re.fullmatch(r'v79_(connected_dark|edge_stroke|consensus|union)_d(\d+)_w([0-9.]+)_a(\d+)_(\d+)',name)
    if not m:raise ValueError(name)
    return m.group(1),{'dark_threshold':int(m.group(2)),'stroke_width_limit':float(m.group(3)),'min_area':int(m.group(4)),'max_area':int(m.group(5))}

V75=Path(V75_RUN_OVERRIDE) if V75_RUN_OVERRIDE else latest_complete(V75_ROOT,['tables/cache_status.csv','tables/surrounding_area_evidence_by_component.csv'])
if V75 is None:raise RuntimeError('No complete controlled V7.5 development run found.')
assert_development_path(V75)
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')
OUT=OUTPUT_ROOT/RUN;TABLES=OUT/'tables';CONTACTS=OUT/'contact_sheets';FINALISTS=OUT/'finalists'
for p in [OUT,TABLES,CONTACTS,FINALISTS]:p.mkdir(parents=True,exist_ok=False)
print('V7.5 reference:',V75);print('Output:',OUT);print('Initial RSS GB:',round(rss_gb(),3))


## Frozen V7.8 baseline and route primitives


In [ ]:
def immutable(original,candidate,mask):
    out=original.copy();out[mask>0]=candidate[mask>0]
    assert np.array_equal(out[mask==0],original[mask==0])
    return out

def background_surface(observed,mask,sigma=45.0):
    visible=(mask==0)&(observed>=200);values=observed.astype(np.float32)*visible;weights=visible.astype(np.float32)
    num=cv2.GaussianBlur(values,(0,0),sigma);den=cv2.GaussianBlur(weights,(0,0),sigma)
    surface=num/np.maximum(den,1e-4);fallback=cv2.GaussianBlur(observed,(0,0),sigma)
    return np.clip(np.where(den>0.02,surface,fallback),0,255).astype(np.uint8)

def support_maps_v78(observed,mask):
    dark=(observed<190)&(mask==0)
    horizontal=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((1,35),np.uint8))>0
    vertical=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((15,1),np.uint8))>0
    n,lab,stats,_=cv2.connectedComponentsWithStats(dark.astype(np.uint8),8);compact=np.zeros_like(dark)
    for i in range(1,n):
        x,y,w,h,a=stats[i]
        if 3<=a<=650 and w<=40 and h<=40:compact[lab==i]=True
    k=np.ones((2*BAND_WIDTH+1,2*BAND_WIDTH+1),np.uint8)
    return cv2.dilate(horizontal.astype(np.uint8),k)>0,cv2.dilate(vertical.astype(np.uint8),k)>0,cv2.dilate(compact.astype(np.uint8),k)>0

def printed_ink(im):return cv2.adaptiveThreshold(im,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0

def page_invariants(observed,mask,telea,clean):
    paper=background_surface(observed,mask);normalised=cv2.divide(observed,np.maximum(paper,1),scale=255)
    horizontal,vertical,compact=support_maps_v78(observed,mask);edges=cv2.Canny(observed,70,170)>0
    clean_ink=printed_ink(clean);clean_edges=cv2.Canny(clean,70,170)>0
    base_preserve=(mask>0)&(horizontal|vertical|compact)&(observed<200)
    base_remove=(mask>0)&~base_preserve&(observed>=GEOMETRY_THRESHOLD)&(normalised>=GEOMETRY_THRESHOLD)
    base_ambiguous=(mask>0)&~base_preserve&~base_remove
    return {'paper':paper,'normalised':normalised,'horizontal':horizontal,'vertical':vertical,'compact':compact,'edges':edges,'clean_ink':clean_ink,'clean_edges':clean_edges,'base_preserve':base_preserve,'base_remove':base_remove,'base_ambiguous':base_ambiguous,'telea':telea}

def render(observed,mask,inv,preserve,remove,ambiguous):
    result=observed.copy();result[remove]=inv['paper'][remove];result[ambiguous]=inv['telea'][ambiguous];result[preserve]=observed[preserve]
    return immutable(observed,result,mask)

def boundary_band(mask,width=3):
    m=(mask>0).astype(np.uint8);inner=cv2.erode(m,np.ones((2*width+1,2*width+1),np.uint8))>0
    return (m>0)&~inner

def connected_map(dark,mask,min_area,max_area,boundary):
    n,lab,stats,_=cv2.connectedComponentsWithStats(dark.astype(np.uint8),8);out=np.zeros_like(dark)
    for i in range(1,n):
        x,y,w,h,a=stats[i];comp=lab==i
        if min_area<=a<=max_area and w<=80 and h<=80 and np.any(comp&boundary):out[comp]=True
    return out

def route_primitives(observed,mask,inv,dark_thresholds,min_areas,max_areas,stroke_widths):
    boundary=boundary_band(mask,3);connected={};edge_stroke={}
    for dt in sorted(set(dark_thresholds)):
        dark=(observed<dt)&(mask>0)
        width=cv2.distanceTransform(dark.astype(np.uint8),cv2.DIST_L2,5)
        dilated_edges=cv2.dilate(inv['edges'].astype(np.uint8),np.ones((3,3),np.uint8))>0
        for sw in sorted(set(stroke_widths)):edge_stroke[(dt,sw)]=dark&dilated_edges&(width<=sw)
        for mina in sorted(set(min_areas)):
            for maxa in sorted(set(max_areas)):
                connected[(dt,mina,maxa)]=connected_map(dark,mask,mina,maxa,boundary)
    return connected,edge_stroke

def glyph_from_primitives(mode,params,connected,edge_stroke):
    c=connected[(params['dark_threshold'],params['min_area'],params['max_area'])]
    e=edge_stroke[(params['dark_threshold'],params['stroke_width_limit'])]
    raw={'connected_dark':c,'edge_stroke':e,'consensus':c&e,'union':c|e}[mode]
    n,lab,stats,_=cv2.connectedComponentsWithStats(raw.astype(np.uint8),8);out=np.zeros_like(raw)
    for i in range(1,n):
        if stats[i,cv2.CC_STAT_AREA]>=params['min_area']:out[lab==i]=True
    return out

def candidate_routes(observed,mask,inv,glyph):
    m=mask>0;preserve=m&(inv['horizontal']|inv['vertical']|inv['compact']|glyph)&(observed<200)
    remove=m&~preserve&(observed>=GEOMETRY_THRESHOLD)&(inv['normalised']>=GEOMETRY_THRESHOLD);ambiguous=m&~preserve&~remove
    return preserve,remove,ambiguous


## Development roster, deterministic discovery subset and V7.8 equivalence


In [ ]:
def image_map(folder):
    assert_development_path(folder)
    return {p.stem:p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if Path(folder).exists() else {}
def loadz(path):
    assert_development_path(path)
    with np.load(path) as z:return {k:z[k] for k in z.files}

clean_map=image_map(ROOT/'source'/'clean');observed_map=image_map(ROOT/'aligned')
status=pd.read_csv(V75/'tables'/'cache_status.csv');cache_lookup={str(r.page):Path(r.cache) for _,r in status.iterrows()}
pages=sorted(set(clean_map)&set(observed_map)&set(cache_lookup))
if PAGE_ALLOWLIST is not None:pages=[p for p in pages if p in set(PAGE_ALLOWLIST)]
if MAX_PAGES is not None:pages=pages[:MAX_PAGES]
if not pages:raise RuntimeError('Controlled development roster is empty.')
discovery_pages=sorted(pages,key=stable_page_key)[:min(DISCOVERY_PAGE_COUNT,len(pages))]
pd.DataFrame({'page':pages,'stable_hash':[stable_page_key(p) for p in pages],'discovery':[p in discovery_pages for p in pages]}).to_csv(TABLES/'development_page_roles.csv',index=False)
print('All development pages:',len(pages));print('Deterministic discovery pages:',len(discovery_pages),discovery_pages)

checks=[]
for page in pages:
    z=loadz(cache_lookup[page]);clean=cv2.imread(str(clean_map[page]),0);observed=cv2.imread(str(observed_map[page]),0)
    for key in ['mask','telea','geometry_t80']:
        if key not in z:raise RuntimeError(f'{page}: missing {key}')
    inv=page_invariants(observed,z['mask'],z['telea'],clean);rebuilt=render(observed,z['mask'],inv,inv['base_preserve'],inv['base_remove'],inv['base_ambiguous'])
    checks.append({'page':page,'geometry_exact':bool(np.array_equal(rebuilt,z['geometry_t80'])),'different_pixels':int(np.sum(rebuilt!=z['geometry_t80'])),'outside_immutable':bool(np.array_equal(rebuilt[z['mask']==0],observed[z['mask']==0]))})
    del z,clean,observed,inv,rebuilt;gc.collect()
baseline_df=pd.DataFrame(checks);baseline_df.to_csv(TABLES/'baseline_equivalence.csv',index=False)
if not baseline_df.geometry_exact.all() or not baseline_df.outside_immutable.all():raise RuntimeError('Frozen V7.8 baseline equivalence failed.')
print('Frozen V7.8 reproduced exactly on all development pages.')


## Route-first scoring

The discovery and confirmation stages compare only additional preserve routing. They do not construct corrected page images.

The score measures:

- additional genuine clean-print pixels preserved;
- additional non-print pixels preserved, treated as false annotation preservation;
- precision of the added glyph route;
- page stability.

This is development scoring only and is not production authorisation.


In [ ]:
def route_score(inv,glyph):
    added=glyph&~inv['base_preserve'];tp=int(np.sum(added&inv['clean_ink']));fp=int(np.sum(added&~inv['clean_ink']))
    precision=tp/max(tp+fp,1);recall_gain=tp/max(int(np.sum(inv['clean_ink'])),1)
    return {'added_pixels':tp+fp,'added_true_print':tp,'added_false_annotation':fp,'added_precision':precision,'clean_print_recall_gain':recall_gain}

def evaluate_route_candidates(selected_pages,candidates,stage):
    rows=[];telemetry=[]
    dts=sorted({p['dark_threshold'] for _,p in candidates});sws=sorted({p['stroke_width_limit'] for _,p in candidates});mins=sorted({p['min_area'] for _,p in candidates});maxs=sorted({p['max_area'] for _,p in candidates})
    for pi,page in enumerate(selected_pages,1):
        z=loadz(cache_lookup[page]);clean=cv2.imread(str(clean_map[page]),0);observed=cv2.imread(str(observed_map[page]),0);mask=z['mask'];inv=page_invariants(observed,mask,z['telea'],clean)
        connected,edge_stroke=route_primitives(observed,mask,inv,dts,mins,maxs,sws)
        for mode,params in candidates:
            glyph=glyph_from_primitives(mode,params,connected,edge_stroke);rows.append({'stage':stage,'page':page,'candidate':candidate_name(mode,params),**route_score(inv,glyph)})
        telemetry.append({'stage':stage,'page':page,'rss_gb':rss_gb(),'candidate_count':len(candidates)})
        pd.DataFrame(rows).to_csv(TABLES/f'{stage}_route_metrics_checkpoint.csv',index=False)
        print(f'{stage}: page {pi}/{len(selected_pages)} {page}; candidates={len(candidates)}; RSS={rss_gb():.3f} GB')
        del z,clean,observed,mask,inv,connected,edge_stroke;gc.collect()
    return pd.DataFrame(rows),pd.DataFrame(telemetry)

def summarise_routes(rows):
    s=rows.groupby('candidate',as_index=False).agg(pages=('page','nunique'),added_true_print=('added_true_print','sum'),added_false_annotation=('added_false_annotation','sum'),mean_precision=('added_precision','mean'),median_precision=('added_precision','median'),mean_recall_gain=('clean_print_recall_gain','mean'),pages_with_gain=('added_true_print',lambda x:int((x>0).sum())),pages_with_false_preservation=('added_false_annotation',lambda x:int((x>0).sum())))
    s['global_precision']=s.added_true_print/(s.added_true_print+s.added_false_annotation).clip(lower=1)
    s['route_utility']=s.mean_recall_gain+0.25*s.global_precision-5*(s.added_false_annotation/(s.added_true_print+s.added_false_annotation).clip(lower=1))
    return s.sort_values(['added_false_annotation','route_utility'],ascending=[True,False])


## Stage 1: coarse route search on deterministic discovery subset


In [ ]:
coarse_candidates=[]
for mode in MODES:
 for dt in COARSE_DARK:
  for sw in COARSE_STROKE:
   for mina in COARSE_MIN_AREA:
    for maxa in COARSE_MAX_AREA:
     coarse_candidates.append((mode,{'dark_threshold':dt,'stroke_width_limit':sw,'min_area':mina,'max_area':maxa}))
coarse_rows,coarse_memory=evaluate_route_candidates(discovery_pages,coarse_candidates,'coarse_discovery')
coarse_summary=summarise_routes(coarse_rows);coarse_summary.to_csv(TABLES/'coarse_route_summary.csv',index=False)
coarse_shortlist=coarse_summary.head(COARSE_KEEP).candidate.tolist()
pd.DataFrame({'candidate':coarse_shortlist,'rank':range(1,len(coarse_shortlist)+1)}).to_csv(TABLES/'coarse_shortlist.csv',index=False)
print('Coarse shortlist:',coarse_shortlist);display(coarse_summary.head(25))


## Stage 2: fine route search around coarse winners


In [ ]:
def neighbours(value,kind):
    if kind=='dark':return sorted({max(60,value-20),max(60,value-10),value,min(220,value+10),min(220,value+20)})
    if kind=='stroke':return sorted({max(1.0,round(value-.75,2)),max(1.0,round(value-.25,2)),value,round(value+.25,2),round(value+.75,2)})
    if kind=='min_area':return sorted({max(1,value-6),max(1,value-3),value,value+3,value+6})
    if kind=='max_area':return sorted({max(100,value-600),max(100,value-300),value,value+300,value+600})

fine_set={}
for name in coarse_shortlist:
    mode,p=parse_candidate(name)
    for dt in neighbours(p['dark_threshold'],'dark'):
     for sw in neighbours(p['stroke_width_limit'],'stroke'):
      for mina in neighbours(p['min_area'],'min_area'):
       for maxa in neighbours(p['max_area'],'max_area'):
        if mina<=maxa:
            q={'dark_threshold':dt,'stroke_width_limit':sw,'min_area':mina,'max_area':maxa};fine_set[candidate_name(mode,q)]=(mode,q)
fine_candidates=list(fine_set.values())
print('Unique fine candidates:',len(fine_candidates))
fine_rows,fine_memory=evaluate_route_candidates(discovery_pages,fine_candidates,'fine_discovery')
fine_summary=summarise_routes(fine_rows);fine_summary.to_csv(TABLES/'fine_route_summary.csv',index=False)
fine_shortlist=fine_summary.head(FINE_KEEP).candidate.tolist()
pd.DataFrame({'candidate':fine_shortlist,'rank':range(1,len(fine_shortlist)+1)}).to_csv(TABLES/'fine_shortlist.csv',index=False)
print('Fine shortlist:',fine_shortlist);display(fine_summary.head(25))


## Stage 3: all-page route confirmation


In [ ]:
confirmation_candidates=[parse_candidate(name) for name in fine_shortlist]
confirm_rows,confirm_memory=evaluate_route_candidates(pages,confirmation_candidates,'all_page_confirmation')
confirm_summary=summarise_routes(confirm_rows);confirm_summary.to_csv(TABLES/'all_page_route_summary.csv',index=False)

# Finalists: first prefer no page with false preservation; otherwise rank conservatively by total false preservation then utility.
strict=confirm_summary[confirm_summary.pages_with_false_preservation==0]
pool=strict if len(strict)>=FINALIST_COUNT else confirm_summary
finalists=pool.head(FINALIST_COUNT).candidate.tolist()
pd.DataFrame({'candidate':finalists,'rank':range(1,len(finalists)+1)}).to_csv(TABLES/'selected_finalists.csv',index=False)
print('Final three candidates:',finalists);display(confirm_summary)


## Stage 4: full reconstruction only for frozen V7.8 and final three


In [ ]:
def tolerant_scores(ref,pred,region,tol=1):
    ref=ref&region;pred=pred&region
    if not ref.sum():return np.nan,np.nan,np.nan
    k=np.ones((2*tol+1,2*tol+1),np.uint8);rd=cv2.dilate(ref.astype(np.uint8),k)>0;pd=cv2.dilate(pred.astype(np.uint8),k)>0
    precision=float(np.sum(pred&rd)/max(pred.sum(),1));recall=float(np.sum(ref&pd)/ref.sum());return precision,recall,float(2*precision*recall/max(precision+recall,1e-12))

def reconstruction_metrics(clean,recon,mask,inv,preserve,remove,glyph):
    m=mask>0;ri=printed_ink(recon);re=cv2.Canny(recon,70,170)>0
    ip,ir,if1=tolerant_scores(inv['clean_ink'],ri,m);ep,er,ef1=tolerant_scores(inv['clean_edges'],re,m)
    diff=np.abs(clean.astype(np.float32)-recon.astype(np.float32));false=int(np.sum(preserve&~inv['clean_ink']));true=int(np.sum(preserve&inv['clean_ink']))
    return {'mae':float(diff[m].mean()),'ink_precision':ip,'ink_recall':ir,'ink_f1':if1,'edge_precision':ep,'edge_recall':er,'edge_f1':ef1,'true_print_preserved':true,'false_annotation_preserved':false,'preserve_precision':true/max(int(preserve.sum()),1),'glyph_pixels':int(glyph.sum()),'removed_clean_print_fraction':float(np.sum(remove&inv['clean_ink']&m)/max(m.sum(),1)),'outside_mask_drift':int(np.sum(recon[~m]!=observed[~m]))}

def save_contact(page,cid,kind,rank,clean,observed,outputs,stats):
    x,y,w,h=stats[cid,:4];margin=55;x0=max(0,x-margin);y0=max(0,y-margin);x1=min(clean.shape[1],x+w+margin);y1=min(clean.shape[0],y+h+margin)
    names=['clean truth','observed','frozen V7.8']+finalists;images=[clean,observed,outputs['v78_baseline']]+[outputs[n] for n in finalists]
    fig,axes=plt.subplots(1,len(images),figsize=(5*len(images),5),squeeze=False)
    for ax,im,name in zip(axes[0],images,names):ax.imshow(im[y0:y1,x0:x1],cmap='gray',vmin=0,vmax=255);ax.set_title(name);ax.axis('off')
    fig.suptitle(f'{kind} #{rank} | {page} component {cid}');plt.tight_layout();folder=CONTACTS/kind;folder.mkdir(parents=True,exist_ok=True);plt.savefig(folder/f'{rank:03d}_{page}_component_{cid:04d}.png',dpi=160,bbox_inches='tight');plt.close(fig)

page_rows=[];component_rows=[];diagnostic_memory=[]
for pi,page in enumerate(pages,1):
    z=loadz(cache_lookup[page]);clean=cv2.imread(str(clean_map[page]),0);observed=cv2.imread(str(observed_map[page]),0);mask=z['mask'];inv=page_invariants(observed,mask,z['telea'],clean)
    final_params=[parse_candidate(n) for n in finalists]
    dts=[p['dark_threshold'] for _,p in final_params];sws=[p['stroke_width_limit'] for _,p in final_params];mins=[p['min_area'] for _,p in final_params];maxs=[p['max_area'] for _,p in final_params]
    connected,edge_stroke=route_primitives(observed,mask,inv,dts or [100],mins or [3],maxs or [650],sws or [2.5])
    outputs={};routes={}
    outputs['v78_baseline']=render(observed,mask,inv,inv['base_preserve'],inv['base_remove'],inv['base_ambiguous']);routes['v78_baseline']=(inv['base_preserve'],inv['base_remove'],inv['base_ambiguous'],np.zeros_like(mask,dtype=bool))
    for name,(mode,params) in zip(finalists,final_params):
        glyph=glyph_from_primitives(mode,params,connected,edge_stroke);preserve,remove,ambiguous=candidate_routes(observed,mask,inv,glyph);outputs[name]=render(observed,mask,inv,preserve,remove,ambiguous);routes[name]=(preserve,remove,ambiguous,glyph)
        np.savez_compressed(FINALISTS/f'{page}_{name}.npz',reconstruction=outputs[name],preserve=preserve.astype(np.uint8),remove=remove.astype(np.uint8),ambiguous=ambiguous.astype(np.uint8),glyph=glyph.astype(np.uint8))
    for name in ['v78_baseline']+finalists:
        preserve,remove,ambiguous,glyph=routes[name];q=reconstruction_metrics(clean,outputs[name],mask,inv,preserve,remove,glyph);page_rows.append({'page':page,'candidate':name,**q})
    n,labels,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8)
    for cid in range(1,n):
        if stats[cid,cv2.CC_STAT_AREA]<MIN_COMPONENT_AREA:continue
        comp=labels==cid
        for name in ['v78_baseline']+finalists:
            recon=outputs[name];preserve,remove,ambiguous,glyph=routes[name];ip,ir,if1=tolerant_scores(inv['clean_ink'],printed_ink(recon),comp);ep,er,ef1=tolerant_scores(inv['clean_edges'],cv2.Canny(recon,70,170)>0,comp)
            component_rows.append({'page':page,'component':cid,'candidate':name,'area':int(comp.sum()),'ink_recall':ir,'ink_f1':if1,'edge_f1':ef1,'mae':float(np.abs(clean.astype(np.float32)-recon.astype(np.float32))[comp].mean()),'false_annotation_preserved':int(np.sum(preserve&comp&~inv['clean_ink']))})
    if finalists:
        local=pd.DataFrame([r for r in component_rows if r['page']==page]);pivot=local.pivot(index='component',columns='candidate',values=['ink_recall','false_annotation_preserved']);best=finalists[0]
        gain=(pivot['ink_recall'][best]-pivot['ink_recall']['v78_baseline']).sort_values(ascending=False).head(MAX_DIAGNOSTIC_COMPONENTS)
        harm=(pivot['false_annotation_preserved'][best]-pivot['false_annotation_preserved']['v78_baseline']).sort_values(ascending=False).head(MAX_DIAGNOSTIC_COMPONENTS)
        for rank,cid in enumerate(gain.index,1):save_contact(page,int(cid),'largest_ink_recall_gains',rank,clean,observed,outputs,stats)
        for rank,cid in enumerate(harm.index,1):save_contact(page,int(cid),'largest_false_preservation_increases',rank,clean,observed,outputs,stats)
    diagnostic_memory.append({'page':page,'rss_gb':rss_gb()});print(f'Full reconstruction {pi}/{len(pages)} {page}; RSS={rss_gb():.3f} GB')
    del z,clean,observed,mask,inv,connected,edge_stroke,outputs,routes,labels,stats;gc.collect()

page_metrics=pd.DataFrame(page_rows);component_metrics=pd.DataFrame(component_rows)
page_metrics.to_csv(TABLES/'finalist_metrics_by_page.csv',index=False);component_metrics.to_csv(TABLES/'finalist_metrics_by_component.csv',index=False)


## Final comparisons and manifest


In [ ]:
summary=page_metrics.groupby('candidate',as_index=False).agg(pages=('page','nunique'),ink_recall=('ink_recall','mean'),ink_f1=('ink_f1','mean'),edge_f1=('edge_f1','mean'),mae=('mae','mean'),false_annotation_preserved=('false_annotation_preserved','sum'),true_print_preserved=('true_print_preserved','sum'),preserve_precision=('preserve_precision','mean'),removed_clean_print_fraction=('removed_clean_print_fraction','mean'),outside_mask_drift=('outside_mask_drift','max'))
base=summary[summary.candidate=='v78_baseline'].iloc[0]
for metric in ['ink_recall','ink_f1','edge_f1','mae','false_annotation_preserved','true_print_preserved','removed_clean_print_fraction']:
    summary[f'delta_{metric}']=summary[metric]-base[metric]
summary.to_csv(TABLES/'finalist_summary.csv',index=False);display(summary)

memory=pd.concat([coarse_memory,fine_memory,confirm_memory,pd.DataFrame(diagnostic_memory).assign(stage='full_reconstruction')],ignore_index=True,sort=False)
memory.to_csv(TABLES/'memory_telemetry.csv',index=False)
manifest={
 'created_utc':datetime.now(timezone.utc).isoformat(),
 'version':'7.9-coarse-to-fine-route-first-3',
 'notebook_name':'22_V7_9_Observed_Ink_Glyph_Preservation_Challenger.ipynb',
 'development_reference':str(V75),
 'all_development_pages':pages,
 'deterministic_discovery_pages':discovery_pages,
 'discovery_selection':'lowest SHA-256 hashes of v79-discovery:<page>',
 'coarse_candidate_count':len(coarse_candidates),
 'fine_candidate_count':len(fine_candidates),
 'all_page_confirmation_count':len(confirmation_candidates),
 'finalists':finalists,
 'route_first_search':True,
 'full_reconstruction_candidates':['v78_baseline']+finalists,
 'v7_8_independent_data_read':False,
 'notebook_09_read_or_modified':False,
 'clean_truth_role':'post-route development scoring and post-reconstruction evaluation only',
 'baseline_equivalence_passed':bool(baseline_df.geometry_exact.all()),
 'selection_is_production_authorisation':False,
 'requires_new_structural_policy':True,
 'requires_new_untouched_independent_validation':True,
 'peak_sampled_rss_gb':float(memory.rss_gb.max()),
}
(OUT/'v7_9_challenger_manifest.json').write_text(json.dumps(manifest,indent=2))
lines=['V7.9 COARSE-TO-FINE ROUTE-FIRST CHALLENGER','',f'All development pages: {len(pages)}',f'Discovery pages: {len(discovery_pages)}',f'Coarse candidates: {len(coarse_candidates)}',f'Fine candidates: {len(fine_candidates)}',f'All-page confirmed candidates: {len(confirmation_candidates)}',f'Finalists: {finalists}',f'Peak sampled RSS: {manifest["peak_sampled_rss_gb"]:.3f} GB','','Development evidence only. Notebook 09 unchanged. V7.8 independent outcomes not read.']
report=chr(10).join(lines);(OUT/'executive_summary.txt').write_text(report);print(report);print('Output:',OUT)
